# DPLL Algorithm Tracer: Understanding Davis-Putnam-Logeman-Loveland

This notebook provides a detailed trace of the DPLL (Davis-Putnam-Logeman-Loveland) algorithm, a backtracking-based SAT solver with powerful optimizations for propositional satisfiability checking.

**Key Learning Objectives:**
- Understand the three core DPLL optimizations
- See how early termination prunes the search space
- Observe pure symbol heuristic in action
- Watch unit clause propagation cascade through assignments
- Compare DPLL efficiency vs exhaustive truth table enumeration

## Section 1: Setup - Import Required Functions

In [1]:
import inspect
import sys

from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent.resolve()))

from utils4e import *
from logic4e import dpll_satisfiable_traced, traced_dpll


# Create test symbols
A, B, C, D, E, F, G, P, Q, R = symbols('A, B, C, D, E, F, G, P, Q, R')

In [2]:
print(inspect.getsource(dpll_satisfiable_traced))

def dpll_satisfiable_traced(s):
    """
    Check satisfiability of a propositional sentence with detailed trace output.
    Shows clause evaluation, pure symbols, unit propagation, and branching.
    """
    clauses = conjuncts(to_cnf(s))
    symbols = list(prop_symbols(s))
    
    print(f"\n{'='*80}")
    print(f"[STARTING dpll_satisfiable_traced]")
    print(f"├─ Original Formula: {s}")
    print(f"├─ CNF Clauses: {len(clauses)} total")
    for i, c in enumerate(clauses, 1):
        print(f"│  {i}. {c}")
    print(f"├─ Symbols to Process: {symbols}")
    print(f"└─ Initial Model: {{}}\n")
    
    result = traced_dpll(clauses, symbols, {})
    
    print(f"\n{'='*80}")
    if result:
        print(f"[FINAL RESULT] SATISFIABLE")
        print(f"├─ Model: {result}")
        print(f"└─ Verification: All clauses evaluate to True with this model")
    else:
        print(f"[FINAL RESULT] UNSATISFIABLE")
        print(f"├─ No valid assignment exists")
        print(f"└─ All possible bran

In [3]:
print(inspect.getsource(traced_dpll))

    def wrapper(clauses, symbols, model):
        
        # increment call stack at each call
        call_depth[0] += 1
        indent = "  " * (call_depth[0] - 1)
        call_id = call_depth[0]
        
        print(f"\n{'='*80}")
        print(f"{indent}[CALL #{call_id}] dpll()")
        print(f"{indent}├─ Recursion Depth: {call_depth[0]}")
        print(f"{indent}├─ Model: {model}")
        print(f"{indent}├─ Remaining Symbols: {symbols}")
        print(f"{indent}└─ Number of Clauses: {len(clauses)}")
        
        # Clause evaluation step
        print(f"{indent}\n[CLAUSE EVALUATION]")
        unknown_clauses = []
        for c in clauses:
            val = pl_true(c, model)
            if val is False:
                print(f"{indent}├─ Clause {c} = FALSE → CONFLICT!")
                call_depth[0] -= 1
                return False
            if val is not True:
                unknown_clauses.append(c)
        
        print(f"{indent}├─ True clauses: {len(clauses) - len(

## Section 2: Understanding DPLL Algorithm Fundamentals



### The DPLL Algorithm Overview

DPLL is a **complete SAT solver** that decides if a Boolean formula in CNF is satisfiable. Unlike truth table enumeration which explores all $2^n$ models, DPLL uses intelligent pruning:

$$\text{dpll(clauses, symbols, model) = } \begin{cases}
\text{True} & \text{if no conflicts remain} \\
\text{False} & \text{if a clause evaluates to False} \\
\text{model} & \text{if all clauses satisfied}
\end{cases}$$

### Three Key Optimizations

**1. Early Termination** - Evaluate clauses with partial model
- $(P \lor Q) \land (P \lor R)$ is TRUE if $P = \text{True}$, regardless of other variables
- If any clause = FALSE → UNSAT (backtrack immediately)

**2. Pure Symbol Heuristic** - Assign symbols appearing only positive or only negative
- Example: $(P \lor \neg Q) \land (\neg Q \lor R)$ has $Q$ as pure (only negative)
- Assign $Q = \text{False}$ without branching

**3. Unit Propagation** - Assign unit clauses (one unbound literal)
- $(P \lor Q) \land Q \land \neg P$ forces $Q = \text{True}$  
- May create new unit clauses (cascading assignments)

## Section 3: The DPLL Trace Decorator

The `trace_dpll` decorator wraps the core `dpll()` function to display:

- **[CALL #n]** - Recursion depth and identification
- **[CLAUSE EVALUATION]** - How many clauses are determined/unknown
- **[CONFLICT]** - When an unsatisfied clause is found
- **[PURE SYMBOL]** - Pure symbol detection and assignment  
- **[UNIT CLAUSE]** - Unit clause forcing and propagation
- **[BRANCH]** - Variable selection and branching
- **[SUCCESS]** - Satisfying assignment found
- **[BACKTRACK]** - Branch exhausted, trying alternative

### Example 1: Simple Satisfiable Formula  

**Formula:** $A \land B \land \neg C \land D$

**Expected:** Satisfiable with model $\{A: T, B: T, C: F, D: T\}$

**What Happens:** Unit clause heuristic immediately assigns all variables without branching.

When a formula is a simple conjunction of literals (unit clauses), each literal directly forces an assignment. DPLL recognizes this instantly and terminates without exploring the decision tree.

In [4]:
formula1 = A & B & ~C & D

print("\n" + "="*80)
print(f"TRACING: {formula1}")
print("="*80)

result1 = dpll_satisfiable_traced(formula1)


TRACING: (((A & B) & ~C) & D)

[STARTING dpll_satisfiable_traced]
├─ Original Formula: (((A & B) & ~C) & D)
├─ CNF Clauses: 4 total
│  1. A
│  2. B
│  3. ~C
│  4. D
├─ Symbols to Process: [C, B, D, A]
└─ Initial Model: {}


[CALL #1] dpll()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [C, B, D, A]
└─ Number of Clauses: 4

[CLAUSE EVALUATION]
├─ True clauses: 0
├─ Unknown clauses: 4

[UNIT CLAUSE] Found unit clause forcing:
├─ Unit variable: A
├─ Forced value: True
└─ Assigning: A = True

  [CALL #2] dpll()
  ├─ Recursion Depth: 2
  ├─ Model: {A: True}
  ├─ Remaining Symbols: [C, B, D]
  └─ Number of Clauses: 4
  
[CLAUSE EVALUATION]
  ├─ True clauses: 1
  ├─ Unknown clauses: 3
  
[UNIT CLAUSE] Found unit clause forcing:
  ├─ Unit variable: B
  ├─ Forced value: True
  └─ Assigning: B = True

    [CALL #3] dpll()
    ├─ Recursion Depth: 3
    ├─ Model: {A: True, B: True}
    ├─ Remaining Symbols: [C, D]
    └─ Number of Clauses: 4
    
[CLAUSE EVALUATION]
    ├─ True clauses

### Example 2: Unsatisfiable Formula (UNSAT)

**Formula:** $(A \lor B) \land \neg A \land \neg B$

**Expected:** Unsatisfiable (False)

**Logic:**
- From $\neg A$: $A = \text{False}$  
- From $\neg B$: $B = \text{False}$
- From $(A \lor B)$: At least one must be True  

**Contradiction!** DPLL quickly detects this impossibility and backtracks.

In [5]:
formula2 = (A | B) & (~A) & (~B)

print("\n" + "="*80)
print(f"TRACING: {formula2}")
print("="*80)

result2 = dpll_satisfiable_traced(formula2)


TRACING: (((A | B) & ~A) & ~B)

[STARTING dpll_satisfiable_traced]
├─ Original Formula: (((A | B) & ~A) & ~B)
├─ CNF Clauses: 3 total
│  1. (A | B)
│  2. ~A
│  3. ~B
├─ Symbols to Process: [B, A]
└─ Initial Model: {}


[CALL #1] dpll()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [B, A]
└─ Number of Clauses: 3

[CLAUSE EVALUATION]
├─ True clauses: 0
├─ Unknown clauses: 3

[UNIT CLAUSE] Found unit clause forcing:
├─ Unit variable: A
├─ Forced value: False
└─ Assigning: A = False

  [CALL #2] dpll()
  ├─ Recursion Depth: 2
  ├─ Model: {A: False}
  ├─ Remaining Symbols: [B]
  └─ Number of Clauses: 3
  
[CLAUSE EVALUATION]
  ├─ True clauses: 1
  ├─ Unknown clauses: 2
  
[UNIT CLAUSE] Found unit clause forcing:
  ├─ Unit variable: B
  ├─ Forced value: True
  └─ Assigning: B = True

    [CALL #3] dpll()
    ├─ Recursion Depth: 3
    ├─ Model: {A: False, B: True}
    ├─ Remaining Symbols: []
    └─ Number of Clauses: 3
    
[CLAUSE EVALUATION]
    ├─ Clause ~B = FALSE → CONFLICT!

### Example 3: Pure Symbol Heuristic

**Formula:** $(A \lor B) \land (B \lor C)$

**Pure Symbols:** $A$ appears only positive (pure literal)

**What Happens:** DPLL identifies $A$ as pure and assigns $A = \text{True}$ without branching. This eliminates an entire subtree of the decision tree.

In [6]:
formula3 = (A | B) & (B | C)

print("\n" + "="*80)
print(f"TRACING: {formula3}")
print(f"Note: A is pure (appears only positive)")
print("="*80)

result3 = dpll_satisfiable_traced(formula3)


TRACING: ((A | B) & (B | C))
Note: A is pure (appears only positive)

[STARTING dpll_satisfiable_traced]
├─ Original Formula: ((A | B) & (B | C))
├─ CNF Clauses: 2 total
│  1. (A | B)
│  2. (B | C)
├─ Symbols to Process: [C, B, A]
└─ Initial Model: {}


[CALL #1] dpll()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [C, B, A]
└─ Number of Clauses: 2

[CLAUSE EVALUATION]
├─ True clauses: 0
├─ Unknown clauses: 2

[PURE SYMBOL] Found pure symbol: C
├─ Appears only as: positive
├─ Auto-assigning: C = True

  [CALL #2] dpll()
  ├─ Recursion Depth: 2
  ├─ Model: {C: True}
  ├─ Remaining Symbols: [B, A]
  └─ Number of Clauses: 2
  
[CLAUSE EVALUATION]
  ├─ True clauses: 1
  ├─ Unknown clauses: 1
  
[PURE SYMBOL] Found pure symbol: B
  ├─ Appears only as: positive
  ├─ Auto-assigning: B = True

    [CALL #3] dpll()
    ├─ Recursion Depth: 3
    ├─ Model: {C: True, B: True}
    ├─ Remaining Symbols: [A]
    └─ Number of Clauses: 2
    
[CLAUSE EVALUATION]
    ├─ True clauses: 2
    ├

### Example 4: Unit Clause Propagation

**Formula:** $(P \lor Q \lor R) \land (\neg P \lor Q \lor R) \land (\neg Q \lor R) \land (\neg R \lor P) \land Q$

**Unit Clause:**  
- $Q$ appears alone as a single clause → forces $Q = \text{True}$

**No Pure Symbols:** Every variable appears in both positive and negative forms:
- $P$: positive in clauses 1,4; negative in clause 2
- $Q$: positive in clauses 1,2,5; negative in clause 3
- $R$: positive in clauses 1,2,3; negative in clause 4

This forces the algorithm to skip pure symbol detection and reach the unit clause check first.

In [7]:
formula4 = (P | Q | R) & (~P | Q | R) & (~Q | R) & (~R | P) & Q

print("\n" + "="*80)
print(f"TRACING: {formula4}")
print("Note: Q is a unit clause (single literal)")
print("All variables appear in both polarities - no pure symbols")
print("="*80)

result4 = dpll_satisfiable_traced(formula4)


TRACING: ((((((P | Q) | R) & ((~P | Q) | R)) & (~Q | R)) & (~R | P)) & Q)
Note: Q is a unit clause (single literal)
All variables appear in both polarities - no pure symbols

[STARTING dpll_satisfiable_traced]
├─ Original Formula: ((((((P | Q) | R) & ((~P | Q) | R)) & (~Q | R)) & (~R | P)) & Q)
├─ CNF Clauses: 5 total
│  1. (P | Q | R)
│  2. (~P | Q | R)
│  3. (~Q | R)
│  4. (~R | P)
│  5. Q
├─ Symbols to Process: [P, Q, R]
└─ Initial Model: {}


[CALL #1] dpll()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [P, Q, R]
└─ Number of Clauses: 5

[CLAUSE EVALUATION]
├─ True clauses: 0
├─ Unknown clauses: 5

[UNIT CLAUSE] Found unit clause forcing:
├─ Unit variable: Q
├─ Forced value: True
└─ Assigning: Q = True

  [CALL #2] dpll()
  ├─ Recursion Depth: 2
  ├─ Model: {Q: True}
  ├─ Remaining Symbols: [P, R]
  └─ Number of Clauses: 5
  
[CLAUSE EVALUATION]
  ├─ True clauses: 3
  ├─ Unknown clauses: 2
  
[UNIT CLAUSE] Found unit clause forcing:
  ├─ Unit variable: R
  ├─ Forced va

### Example 5: Complex Formula with Multiple Heuristics

**Formula:** $(A \lor B \lor C) \land (\neg A \lor D) \land (\neg B \lor \neg D) \land E$

**Heuristics Applied:**
- **Unit clause:** $E$ (forces $E = \text{True}$)
- **Pure symbols:** May be detected after unit propagation
- **Branching:** When no pure symbols or units exist

This realistic formula demonstrates all three heuristics working together.

In [ ]:
formula5 = (A | B | C) & (~A | D) & (~B | ~D) & E # type: ignore

print("\n" + "="*80)
print(f"TRACING: {formula5}")
print("Multiple heuristics will apply")
print("="*80)

result5 = dpll_satisfiable_traced(formula5)


TRACING: (((((A | B) | C) & (~A | D)) & (~B | ~D)) & E)
Multiple heuristics will apply

[STARTING dpll_satisfiable_traced]
├─ Original Formula: (((((A | B) | C) & (~A | D)) & (~B | ~D)) & E)
├─ CNF Clauses: 4 total
│  1. (A | B | C)
│  2. (~A | D)
│  3. (~B | ~D)
│  4. E
├─ Symbols to Process: [A, E, D, C, B]
└─ Initial Model: {}


[CALL #1] dpll()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [A, E, D, C, B]
└─ Number of Clauses: 4

[CLAUSE EVALUATION]
├─ True clauses: 0
├─ Unknown clauses: 4

[UNIT CLAUSE] Found unit clause forcing:
├─ Unit variable: E
├─ Forced value: True
└─ Assigning: E = True

  [CALL #2] dpll()
  ├─ Recursion Depth: 2
  ├─ Model: {E: True}
  ├─ Remaining Symbols: [A, D, C, B]
  └─ Number of Clauses: 4
  
[CLAUSE EVALUATION]
  ├─ True clauses: 1
  ├─ Unknown clauses: 3
  
[PURE SYMBOL] Found pure symbol: C
  ├─ Appears only as: positive
  ├─ Auto-assigning: C = True

    [CALL #3] dpll()
    ├─ Recursion Depth: 3
    ├─ Model: {E: True, C: True}
    ├

## Section 4: The Heuristic Ordering Problem - Why Unit Clauses Must Come First



### The Current Implementation Issue

The DPLL algorithm checks heuristics in this order:
```
1. Find Pure Symbols        ← This is checked first
2. Find Unit Clauses        ← This is checked second
3. Pick and Branch          ← Last resort
```

**This is backwards!** The correct order should be:
```
1. Find Unit Clauses        ← FORCED assignments (must happen)
2. Find Pure Symbols        ← OPTIMIZATION (prunes branches)
3. Pick and Branch          ← Last resort search
```

### Why This Matters: Key Reasons

#### 1. Computational Efficiency

**Time Complexity Analysis:**

- **Unit Clause Detection:** O(n) where n = number of clauses
  ```
  Scan each clause once for unbound literals → Quick!
  ```

- **Pure Symbol Detection:** O(n·m) where m = number of symbols
  ```
  For each symbol: scan all clauses to check polarity
  Nested loops make this quadratic
  ```

**The Problem:** When unit clauses exist, checking pure symbols first wastes O(n·m) time before finding the forced assignment.

**Example with numbers:**
- Formula with 10 variables, 4 clauses after assignment
- Pure symbol check: O(3 × 4) = 12 operations
- Then unit clause check: O(4) operations
- **Total wasted:** 12 when we could have done 4 in reverse order!

#### 2. Logical Correctness

**Unit Clauses = MANDATORY Assignments:**
```
Clause: (~Q | R) after Q = True
Becomes: (False | R) = R    ← Single literal, must assign R=True
```

No choice exists. If the formula is satisfiable, this assignment is **required**.

**Pure Symbols = OPTIONAL OPTIMIZATIONS:**
```
Symbol P appears only positive: (P | Q), (P | R)

If P = True: Clauses satisfied ✓
If P = False: Still satisfiable (depends on Q, R)

Pure symbol heuristic is a shortcut, not a requirement.
```

**The Principle:** Always resolve forced assignments before optimizations.

#### 3. Search Space Reduction

**Cascading Simplification:**

After Unit Clause Q = True:
```
Original:  (P | Q | R) & (~P | Q | R) & (~Q | R) & (~R | P)
Becomes:   TRUE & TRUE & R & (~R | P)
Simplifies: R & (~R | P)
```

Now R is a new unit clause! This creates a cascade:
```
Assign R = True
  ↓
(~R | P) becomes P
  ↓
P is new unit clause!
  ↓
Assign P = True
```

**One decision → Three automatic assignments!**

If we check pure symbols first, we miss this cascading opportunity.

#### 4. Following CS Standards

All modern SAT solvers use this order:

| Solver | Standard | Order |
|--------|----------|-------|
| **DPLL (1962)** | Original | Unit first |
| **CDCL (2000s)** | State-of-the-art | Unit first |
| **CaDiCaL, MapleSAT** | Competition winners | Unit first |
| **Kissat, Cryptominisat** | Modern solvers | Unit first |

Academic consensus: **Unit propagation is the "engine" of SAT solving.**

#### 5. Code Intent & Teaching

**Reading the code should reveal intent:**
```python
# Good order:
unit_clause, value = find_unit_clause(clauses, model)
if unit_clause:
    return dpll(...)  # "Forced assignment - must happen"

pure_symbol, value = find_pure_symbol(symbols, clauses, model)
if pure_symbol:
    return dpll(...)  # "Optimization - only after forcing"
```

This teaches students the correct principle: **Forcing before optimizing.**

In [9]:
comparison_text = """
═══════════════════════════════════════════════════════════════════════════════
SCENARIO: After Q = True, these clauses remain:

Remaining Unsatisfied Clauses:
├─ (P | T | R) = TRUE        ✓ Already satisfied
├─ (~P | T | R) = TRUE       ✓ Already satisfied
├─ (F | R) = R               ← UNIT CLAUSE! (Single literal)
└─ (~R | P)                  ← Mixed literals

═══════════════════════════════════════════════════════════════════════════════

STRATEGY A: Check Pure FIRST (INEFFICIENT - Current Implementation)
──────────────────────────────────────────────────────────────────

Step 1: find_pure_symbol(symbols, clauses, model)  [O(n·m) = O(3·4)]
├─ Scan symbol P:
│  ├─ Appears positive in clause: (~R | P)  ✓
│  ├─ Appears negative in clause: Not found in remaining  ✓
│  └─ P is PURE (positive only)!
├─ Decision: Assign P = True
└─ Result: One assignment, no cascade

Step 2: find_unit_clause(clauses, model)          [O(n) = O(4)]
├─ Scan clauses for single unbound literal
├─ Found: Clause 3 is just R
└─ Result: Unit R found (but we already made a decision)

Outcome: Checks pure first (wasted O(n·m)) then finds unit
Time: O(n·m) + O(n) = QUADRATIC

═══════════════════════════════════════════════════════════════════════════════

STRATEGY B: Check Unit FIRST (EFFICIENT - Correct Implementation)
────────────────────────────────────────────────────────────────

Step 1: find_unit_clause(clauses, model)          [O(n) = O(4)]
├─ Scan clause 1: TRUE (skip)
├─ Scan clause 2: TRUE (skip)
├─ Scan clause 3: (F | R) = R  ← FOUND UNIT!
└─ Decision: Assign R = True

Step 2: Clause simplification (automatic)
├─ Original clause 4: (~R | P)
├─ After R=True: (F | P) = P
└─ P becomes NEW UNIT CLAUSE!

Step 3: find_unit_clause(clauses, model) again    [O(n) = O(4)]
├─ Found: P is now a unit clause
└─ Decision: Assign P = True (cascading!)

Outcome: One pure symbol check avoided, cascading simplification
Time: O(n) + O(n) = LINEAR

═══════════════════════════════════════════════════════════════════════════════

CASCADING EFFECT - The Real Win:

Unit First Order:
─────────────────
Recursion Depth 1:
  Decision 1: Q = True
    └─ Triggers cascade: R becomes unit
      └─ Decision 2: R = True
        └─ Triggers cascade: P becomes unit
          └─ Decision 3: P = True
              └─ All constraints satisfied!

Result: 3 variables assigned in sequential depth (pseudo-linear)
Decision tree size: O(3) sequential assignments

Pure First Order:
─────────────────
Recursion Depth 1:
  Decision 1: P = True (pure detection costs O(n·m))
    └─ Remaining: Q, R still need decisions
Recursion Depth 2:
  Decision 2: Must pick Q, R arbitrarily
    └─ Requires branching: 2^2 = 4 possibilities
      └─ Each branch needs unit check

Result: Forced to branch, exponential exploration
Decision tree size: O(2^n) exponential

═══════════════════════════════════════════════════════════════════════════════

COMPLEXITY COMPARISON - 10 Variable Formula:

With effective unit clause cascade (Horn-like):

Pure First:                          Unit First:
───────────                          ──────────
Depth 1: Check pure O(10·4)          Depth 1: Check unit O(4)
         No pure found               → Assign Q, R cascade
         Must pick, branch 2^8       
                                     Depth 2: Check unit O(4)
Depth 2-9: Exponential branching     → Assign P, S cascade
         Explore ~256 nodes          
                                     Depth 3: Only 5 vars left
Total: ~256 nodes                    Depth 4-5: 2^5 = 32 nodes

SPEEDUP: 256 / 32 = 8× faster!

═══════════════════════════════════════════════════════════════════════════════
"""

print(comparison_text)


═══════════════════════════════════════════════════════════════════════════════
SCENARIO: After Q = True, these clauses remain:

Remaining Unsatisfied Clauses:
├─ (P | T | R) = TRUE        ✓ Already satisfied
├─ (~P | T | R) = TRUE       ✓ Already satisfied
├─ (F | R) = R               ← UNIT CLAUSE! (Single literal)
└─ (~R | P)                  ← Mixed literals

═══════════════════════════════════════════════════════════════════════════════

STRATEGY A: Check Pure FIRST (INEFFICIENT - Current Implementation)
──────────────────────────────────────────────────────────────────

Step 1: find_pure_symbol(symbols, clauses, model)  [O(n·m) = O(3·4)]
├─ Scan symbol P:
│  ├─ Appears positive in clause: (~R | P)  ✓
│  ├─ Appears negative in clause: Not found in remaining  ✓
│  └─ P is PURE (positive only)!
├─ Decision: Assign P = True
└─ Result: One assignment, no cascade

Step 2: find_unit_clause(clauses, model)          [O(n) = O(4)]
├─ Scan clauses for single unbound literal
├─ Found: Claus

## Section 5: Reading the Trace Output Guide



### Key Output Markers

| Marker | Meaning |
|--------|---------|
| `[CALL #n]` | Recursive call at depth n |
| `[CLAUSE EVALUATION]` | Checking all clauses in current model |
| `[SUCCESS]` | All clauses satisfied |
| `[CONFLICT]` | Found unsatisfied clause → backtrack |
| `[PURE SYMBOL]` | Pure symbol detected and assigned |
| `[UNIT CLAUSE]` | Unit clause found, variable forced |
| `[BRANCH]` | Pick variable, explore both values |
| `[BACKTRACK]` | Branch failed, trying alternative |

### Understanding Indentation

```
Depth 0: [CALL #1]         ← Root call
  Depth 1: [CALL #2]       ← 2 spaces = first decision
    Depth 2: [CALL #3]     ← 4 spaces = second decision
      Depth 3: [CALL #4]   ← 6 spaces = third decision
```

Deeper indentation = deeper in the decision tree.

## Section 6: DPLL vs Truth Table Enumeration - Complexity Analysis



### Comparison

| Aspect | Truth Table | DPLL |
|--------|------------|------|
| Explores all $2^n$ models? | Yes, always | No, prunes branches |
| Early termination? | No | Yes, detects conflicts |
| Pure symbol optimization? | N/A | Yes |
| Unit propagation? | N/A | Yes |
| Practical for $n > 20$? | No | Often yes |

### Worst-Case Complexity

- **Truth Table:** Always $O(2^n)$ evaluations
- **DPLL:** Still $O(2^n)$ worst case, but exponentially better on average

### When DPLL Shines

**Best case (linear time):**
- Pure symbol dominated: assign without branching
- Unit clause dominated: cascade assignments
- Example: Horn clauses with facts

**Worst case (exponential):**
- Random 3-SAT formulas near phase transition
- Balanced clauses, no pure symbols
- Many conflicts at different depths

### Real-World Impact

Modern SAT solvers (CDCL) build on DPLL:
- **Learned clauses** remember failed branches
- **Non-chronological backtracking** jumps over irrelevant decisions  
- **Variable activity** picks "important" symbols first
- **Restarts** escape local search patterns

These enhancements allow solving problems with 1,000,000+ variables!

## Summary: What You've Learned



✓ **Three DPLL optimizations:** Early termination, pure symbols, unit propagation  
✓ **How DPLL avoids exponential explosion:** Intelligent pruning reduces search space  
✓ **Pure symbol heuristic in action:** Eliminating symmetric branches  
✓ **Unit propagation cascades:** Assignments create new forced assignments  
✓ **When conflicts occur:** Backtracking to explore alternatives  
✓ **Trace interpretation:** Reading recursion depth, heuristic applications, decisions  
✓ **Complexity analysis:** Why DPLL matters compared to exhaustive search  
✓ **Real-world SAT solvers:** Modern enhancements beyond basic DPLL  

### Key Insight

DPLL demonstrates a fundamental principle of **intelligent search**: domain knowledge (pure symbols, unit clauses) can reduce exponential problems to practical levels. While worst-case complexity remains exponential, the combination of three simple heuristics makes DPLL practical for many real-world problems.

### Next Steps

- Experiment with different formulas in the code cells
- Observe which heuristic activates for each formula  
- Compare execution traces to understand search efficiency
- Try formulas at the "phase transition" (hardest to solve)

---

**Reference:** Chapter 7 (Logical Agents), AIMA 4th Edition  
**Key Paper:** Davis, Putnam, Logemann, Loveland (1962) - "A Machine Program for Theorem-Proving"